In [52]:

import pandas as pd
# đọc file
df1 = pd.read_csv("trade balance.csv")
df2 = pd.read_csv("Macro_Vietnam(Monthly).csv")
df3 = pd.read_csv("FEDFUNDS.csv")
df4 = pd.read_csv("CPI VN.csv")
df5 = pd.read_csv("Dữ liệu Lịch sử USD_VND.csv")

In [53]:
import re

In [54]:
import numpy as np

In [55]:


df4 = df4.rename(columns={
    "TIME_PERIOD": "date",
    "OBS_VALUE": "CPI"
})

df4["date"] = pd.to_datetime(df4["date"].str.replace("M", "", regex=False), format="%Y-%m")

# format lại
df4["date"] = df4["date"].dt.strftime("%m-%Y")

print(df4.head())

      date        CPI
0  02-2016  88.827799
1  03-2016  89.333673
2  04-2016  89.632763
3  05-2016  90.117497
4  06-2016  90.536092


In [56]:


df5 = df5[["Ngày", "Lần cuối"]].rename(columns={
    "Ngày": "date",
    "Lần cuối": "USD/VND"
})

# xử lý dấu phẩy trong số (26,045.0 → 26045.0)
df5["USD/VND"] = df5["USD/VND"].str.replace(",", "", regex=False).astype(float)

# convert date dd/mm/yyyy → datetime
df5["date"] = pd.to_datetime(df5["date"], format="%d/%m/%Y", errors="coerce")

# lấy tháng-năm
df5["date"] = df5["date"].dt.strftime("%m-%Y")
print(df5.head())

      date  USD/VND
0  02-2026  26045.0
1  01-2026  25940.0
2  12-2025  26300.0
3  11-2025  26365.0
4  10-2025  26315.0


In [57]:
# tạo cột datetime để sort
df5["date_sort"] = pd.to_datetime(df5["date"], format="%m-%Y")

# sort tăng dần
df5 = df5.sort_values("date_sort")

# xoá cột phụ
df5 = df5.drop(columns=["date_sort"])

# reset index
df5 = df5.reset_index(drop=True)

In [58]:
print(df5.head())

      date  USD/VND
0  02-2016  22277.0
1  03-2016  22264.0
2  04-2016  22244.0
3  05-2016  22366.0
4  06-2016  22281.0


In [59]:

df3 = df3[["observation_date", "FEDFUNDS"]].rename(columns={
    "observation_date": "date",
})

# convert sang datetime
df3["date"] = pd.to_datetime(df3["date"], format="%Y-%m-%d", errors="coerce")

# lấy tháng-năm
df3["date"] = df3["date"].dt.strftime("%m-%Y")

print(df3.head())

      date  FEDFUNDS
0  02-2016      0.38
1  03-2016      0.36
2  04-2016      0.37
3  05-2016      0.37
4  06-2016      0.38


In [60]:


df2 = df2[["Period", "Interest rate (12-24 month)"]].rename(columns={
    "Period": "date",
    "Interest rate (12-24 month)": "VN_IR"
})

# convert date
df2["date"] = pd.to_datetime(df2["date"], format="%m-%Y", errors="coerce")

# xử lý %
df2["VN_IR"] = df2["VN_IR"].str.replace("%", "", regex=False).astype(float) / 100

# sort theo thời gian
df2 = df2.sort_values("date")

# 🔥 FORMAT LẠI DATE → MM-YYYY
df2["date"] = df2["date"].dt.strftime("%m-%Y")

# reset index
df2 = df2.reset_index(drop=True)

print(df2.head())

      date   VN_IR
0  01-2010  0.1049
1  02-2010  0.1049
2  03-2010  0.1049
3  04-2010  0.1130
4  05-2010  0.1150


In [61]:

df1 = df1[["Release date", "Actual"]].rename(columns={
    "Release date": "date",
    "Actual": "VN_TB"
})


def parse_date(x):
    if pd.isna(x):
        return np.nan
    
    x = str(x)
    
    # case 1: "Apr 06, 2026 (Mar)"
    if "(" in x:
        month_abbr = re.search(r"\((\w{3})\)", x).group(1)
        year = re.search(r"\d{4}", x).group()
        return pd.to_datetime(f"{month_abbr} {year}", format="%b %Y")
    
    # case 2: "2024-06-29 00:00:00"
    else:
        return pd.to_datetime(x, errors="coerce")

# apply
df1["date"] = df1["date"].apply(parse_date)

# lấy MM-YYYY
df1["date"] = df1["date"].dt.strftime("%m-%Y")

# =========================
# 🔥 2. XỬ LÝ VALUE (VN_TB)
# =========================

def clean_value(x):
    if pd.isna(x):
        return np.nan
    
    x = str(x)
    
    # bỏ dấu phẩy
    x = x.replace(",", "")
    
    # check đơn vị M (million)
    if "M" in x:
        x = x.replace("M", "")
        return float(x) * 1_000_000
    
    return float(x)

df1["VN_TB"] = df1["VN_TB"].apply(clean_value)

# =========================
# 🔥 3. SORT
# =========================

df1 = df1.sort_values("date").reset_index(drop=True)

print(df1.head())

      date         VN_TB
0  01-2018 -3.000000e+08
1  01-2019 -8.000000e+08
2  01-2020  2.590000e+08
3  01-2021  1.300000e+09
4  01-2022 -5.000000e+08


In [67]:
df1 = df1.groupby("date", as_index=False).mean()

In [68]:
df1["date"].duplicated().sum()

np.int64(0)

In [69]:
# 1. xử lý duplicate từng file
df1 = df1.groupby("date", as_index=False).mean()
df2 = df2.groupby("date", as_index=False).mean()
df3 = df3.groupby("date", as_index=False).mean()
df4 = df4.groupby("date", as_index=False).mean()
df5 = df5.groupby("date", as_index=False).mean()

# 2. merge
from functools import reduce
dfs = [df1, df2, df3, df4, df5]

df_final = reduce(lambda left, right: pd.merge(left, right, on="date", how="inner"), dfs)

In [70]:
from functools import reduce
import pandas as pd

# list các dataframe của bạn
dfs = [df1, df2, df3, df4, df5]

# 🔥 merge theo phần giao nhau
df_final = reduce(lambda left, right: pd.merge(left, right, on="date", how="inner"), dfs)

# sort lại theo thời gian
df_final = df_final.sort_values("date").reset_index(drop=True)

print(df_final.head())
print(df_final.shape)

      date         VN_TB  VN_IR  FEDFUNDS         CPI  USD/VND
0  01-2018 -3.000000e+08  0.073      1.41   95.537483  22700.0
1  01-2019 -8.000000e+08  0.073      2.40   97.986081  23198.0
2  01-2020  2.590000e+08  0.075      1.55  104.284008  23222.0
3  01-2021  1.300000e+09  0.068      0.09  103.271124  23048.0
4  01-2022 -5.000000e+08  0.065      0.08  105.276475  22645.0
(97, 6)


In [73]:
df_final["VN_IR"] = df_final["VN_IR"].round(3)

In [74]:
# 🔥 convert về datetime
df_final["date"] = pd.to_datetime(df_final["date"], format="%m-%Y")

# 🔥 sort đúng theo thời gian
df_final = df_final.sort_values("date")

# 🔥 nếu bạn muốn hiển thị lại dạng MM-YYYY
df_final["date"] = df_final["date"].dt.strftime("%m-%Y")

# reset index
df_final = df_final.reset_index(drop=True)

print(df_final.head(20))

       date         VN_TB  VN_IR  FEDFUNDS        CPI  USD/VND
0   12-2016 -3.000000e+08  0.072      0.54  92.649082  22770.0
1   02-2017 -6.500000e+08  0.072      0.66  93.289534  22750.0
2   03-2017 -1.100000e+09  0.072      0.79  93.483576  22750.0
3   04-2017 -8.000000e+08  0.072      0.90  93.486568  22740.0
4   05-2017 -8.000000e+08  0.072      0.91  92.993480  22710.0
5   06-2017 -2.000000e+08  0.072      1.04  92.834368  22700.0
6   07-2017 -3.000000e+08  0.072      1.15  92.936207  22726.0
7   08-2017  4.000000e+08  0.072      1.16  93.788618  22715.0
8   09-2017  4.000000e+08  0.072      1.15  94.343940  22720.0
9   10-2017  9.000000e+08  0.072      1.15  94.734147  22700.0
10  11-2017  2.000000e+08  0.072      1.16  94.858722  22705.0
11  12-2017 -2.340000e+08  0.073      1.30  95.054511  22700.0
12  01-2018 -3.000000e+08  0.073      1.41  95.537483  22700.0
13  02-2018  9.000000e+08  0.073      1.42  96.232231  22756.0
14  03-2018  8.000000e+08  0.073      1.51  95.973270  

In [ ]:
df_final.to_csv(r"C:\Users\ADMIN\Documents\time series\forecast_USD_VND.csv", index=False, encoding="utf-8-sig")

: 